# Ensemble Machine Learning Based Bank Loan Approval Prediction System

**Course / Project**: Academic Machine Learning Capstone  
**Objective**: Develop, evaluate, and interpret an ensemble learning architecture combining Random Forest, Extra Trees, and K-Nearest Neighbors alongside six benchmark classifiers for risk assessment and loan approval prediction.  
**Data Integrity Note**: This dataset and experiment use synthetic academic data generated with realistic probabilistic underwriting distributions to study ML performance without exposing proprietary customer records.

## 1. Environment Setup & Library Installations
Execute the cell below if running in Google Colab or a fresh local environment.

In [ ]:
# Colab installation command
!pip install -q pandas numpy scikit-learn imbalanced-learn matplotlib seaborn plotly joblib

## 2. Core Library Imports
We import foundational numerical, statistical, machine learning, and visualization libraries.

In [ ]:
import os
import sys
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Imbalanced-Learn for SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

print("All core dependencies loaded successfully.")

## 3. Dataset Loading & Academic Fallback Generator
We inspect whether the dataset CSV exists locally. If running in Google Colab without mounted storage, the cell automatically synthesizes the academic dataset using the documented underwriting formula.

In [ ]:
DATA_PATH = "../data/loan_dataset.csv" if os.path.exists("../data/loan_dataset.csv") else "loan_dataset.csv"

if not os.path.exists(DATA_PATH):
    print("Local dataset not found. Generating compliant synthetic academic dataset...")
    rng = np.random.default_rng(42)
    n_samples = 5200
    genders = rng.choice(["Male", "Female"], size=n_samples, p=[0.78, 0.22])
    married = rng.choice(["Yes", "No"], size=n_samples, p=[0.65, 0.35])
    dependents = [rng.choice(["0", "1", "2", "3"], p=[0.5, 0.25, 0.18, 0.07]) for _ in range(n_samples)]
    education = rng.choice(["Graduate", "Not Graduate"], size=n_samples, p=[0.78, 0.22])
    self_employed = rng.choice(["No", "Yes"], size=n_samples, p=[0.86, 0.14])
    property_area = rng.choice(["Semiurban", "Urban", "Rural"], size=n_samples, p=[0.38, 0.33, 0.29])
    
    app_inc = np.clip(np.exp(rng.normal(8.3, 0.6, n_samples)).astype(int), 1200, 85000)
    has_coapp = rng.choice([0, 1], size=n_samples, p=[0.42, 0.58])
    coapp_inc = np.where(has_coapp == 1, np.clip(np.exp(rng.normal(7.6, 0.7, n_samples)).astype(int), 800, 40000), 0)
    tot_inc = app_inc + coapp_inc
    
    loan_term = rng.choice([180, 240, 300, 360, 480], size=n_samples, p=[0.08, 0.04, 0.02, 0.84, 0.02])
    loan_amount = np.clip((tot_inc * 0.024) + rng.normal(30, 25, n_samples), 15, 750).astype(int)
    
    cred_prob = np.clip(0.75 + (education == "Graduate")*0.05 + (tot_inc > 6000)*0.06 - (loan_amount > 350)*0.05, 0.1, 0.95)
    credit_history = rng.binomial(1, cred_prob, n_samples).astype(float)
    
    debt_burden = ((loan_amount * 1000) / loan_term) / np.maximum(tot_inc, 1000)
    latent = -1.5 + 3.2*credit_history - 2.8*np.clip(debt_burden - 0.28, 0, 1) + 0.5*(property_area == "Semiurban") + 0.35*(np.log(tot_inc)-8.5) + rng.logistic(0, 0.65, n_samples)
    loan_status = np.where(1 / (1 + np.exp(-latent)) >= 0.5, "Y", "N")
    
    df = pd.DataFrame({
        "Loan_ID": [f"LP{str(i+100001).zfill(6)}" for i in range(n_samples)],
        "Gender": genders, "Married": married, "Dependents": dependents, "Education": education,
        "Self_Employed": self_employed, "ApplicantIncome": app_inc, "CoapplicantIncome": coapp_inc,
        "LoanAmount": loan_amount, "Loan_Amount_Term": loan_term, "Credit_History": credit_history,
        "Property_Area": property_area, "Loan_Status": loan_status
    })
    df.to_csv("loan_dataset.csv", index=False)
    DATA_PATH = "loan_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")
display(df_raw.head())

## 4. Exploratory Data Analysis & Target Balance
Understanding the distribution of the target class (`Loan_Status`) and feature relationships.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
sns.countplot(data=df_raw, x="Loan_Status", palette="crest")
plt.title("Loan Approval Distribution")

plt.subplot(1, 3, 2)
sns.countplot(data=df_raw, x="Credit_History", hue="Loan_Status", palette="viridis")
plt.title("Approval by Credit History")

plt.subplot(1, 3, 3)
sns.countplot(data=df_raw, x="Property_Area", hue="Loan_Status", palette="mako")
plt.title("Approval by Property Area")
plt.tight_layout()
plt.show()

## 5. Feature Engineering & Preprocessing
We compute domain-specific debt burden indicators, construct ColumnTransformers, and isolate the training/test split.

In [ ]:
# Deduplication
df_clean = df_raw.drop_duplicates(subset=["Loan_ID"]).copy()

# Feature engineering
tot_inc = df_clean["ApplicantIncome"].fillna(0) + df_clean["CoapplicantIncome"].fillna(0)
df_clean["Total_Income"] = tot_inc
df_clean["Loan_Income_Ratio"] = (df_clean["LoanAmount"] * 1000) / np.maximum(tot_inc, 1000)
term = np.maximum(df_clean["Loan_Amount_Term"].fillna(360), 12)
df_clean["Debt_Burden"] = ((df_clean["LoanAmount"] * 1000) / term) / np.maximum(tot_inc, 1000)
df_clean["Dependents"] = df_clean["Dependents"].astype(str).str.replace("3+", "3")

NUM_COLS = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term", "Credit_History", "Total_Income", "Loan_Income_Ratio", "Debt_Burden"]
CAT_COLS = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]

X = df_clean[NUM_COLS + CAT_COLS]
y = df_clean["Loan_Status"].map({"N": 0, "Y": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 6. Model Training & Cross-Validation with SMOTE Pipeline
We train all 9 baseline classifiers plus the soft-voting ensemble (`Random Forest` + `Extra Trees` + `K-Nearest Neighbors`).

In [ ]:
def create_pipeline(clf):
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    preprocessor = ColumnTransformer([("num", num_pipe, NUM_COLS), ("cat", cat_pipe, CAT_COLS)])
    return ImbPipeline([("prep", preprocessor), ("smote", SMOTE(random_state=42)), ("clf", clf)])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=9, weights="distance"),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=120, random_state=42),
    "Voting Ensemble": VotingClassifier([
        ("rf", RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)),
        ("et", ExtraTreesClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)),
        ("knn", KNeighborsClassifier(n_neighbors=9, weights="distance"))
    ], voting="soft")
}

results = []
fitted_models = {}

for name, clf in models.items():
    pipe = create_pipeline(clf)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    fitted_models[name] = pipe

metrics_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(metrics_df)

## 7. Comparative Model Performance & Confusion Matrix
Visualizing ROC curves and classification metrics.

In [ ]:
plt.figure(figsize=(10, 6))
for name in ["Voting Ensemble", "Random Forest", "Extra Trees", "KNN", "Logistic Regression"]:
    y_prob = fitted_models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Baseline (AUC = 0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Comparison")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Single Applicant Prediction Test
Testing inference on a new applicant profile.

In [ ]:
sample_applicant = pd.DataFrame([{
    "ApplicantIncome": 6200,
    "CoapplicantIncome": 2100,
    "LoanAmount": 145,
    "Loan_Amount_Term": 360,
    "Credit_History": 1.0,
    "Total_Income": 8300,
    "Loan_Income_Ratio": (145 * 1000) / 8300,
    "Debt_Burden": ((145 * 1000) / 360) / 8300,
    "Gender": "Male",
    "Married": "Yes",
    "Dependents": "1",
    "Education": "Graduate",
    "Self_Employed": "No",
    "Property_Area": "Semiurban"
}])

ensemble_model = fitted_models["Voting Ensemble"]
pred_class = ensemble_model.predict(sample_applicant)[0]
pred_proba = ensemble_model.predict_proba(sample_applicant)[0]

print(f"Predicted Decision: {'APPROVED (Y)' if pred_class == 1 else 'REJECTED (N)'}")
print(f"Approval Probability: {pred_proba[1]*100:.2f}%")
print(f"Rejection Probability: {pred_proba[0]*100:.2f}%")

## 9. Academic Conclusions & Limitations
- **Underwriting Primacy**: Consistent with banking literature, credit history and debt-to-income ratio represent the strongest individual indicators of repayment capability.
- **Ensemble Benefit**: Soft voting reduces variance across base estimators, producing smoother decision boundaries and calibrated probability estimates.
- **Limitations**: This is an academic study using synthetic training patterns. Real lending systems must comply with fair-lending laws (e.g. ECOA, FCRA), explainability regulations, and periodic auditing for demographic parity.